In [21]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [4]:
df_job = pd.read_csv('../data/job_title_des.csv')
df_resume = pd.read_csv('../data/Resume.csv')

In [5]:
df_job["bag of words"] = df_job["Job Description"].map(
    lambda x: [t for t in re.split(r"[ \n.,!?;:()\[\]{}\"'’\-_/]+", x.lower()) if t]
)

df_job['bag of words']

0       [we, are, looking, for, hire, experts, flutter...
1       [python, django, developer, lead, job, code, p...
2       [data, scientist, contractor, bangalore, in, r...
3       [job, description, strong, framework, outside,...
4       [job, responsibility, full, stack, engineer, –...
                              ...                        
2272    [job, summary, published, on, 26, days, ago, v...
2273    [business, entity, cisco, umbrella, focus, clo...
2274    [urgently, reqd, in, a, college, in, mohali, n...
2275    [key, responsibilities, team, leads, for, smal...
2276    [leslie, hindman, auctioneer, one, nation, s, ...
Name: bag of words, Length: 2277, dtype: object

In [6]:
df_resume["bag of words"] = df_resume["Resume_str"].map(
    lambda x: [t for t in re.split(r"[ \n.,!?;:()\[\]{}\"'’\-_/]+", x.lower()) if t]
)

df_resume['bag of words']

0       [hr, administrator, marketing, associate, hr, ...
1       [hr, specialist, us, hr, operations, summary, ...
2       [hr, director, summary, over, 20, years, exper...
3       [hr, specialist, summary, dedicated, driven, a...
4       [hr, manager, skill, highlights, hr, skills, h...
                              ...                        
2479    [rank, sgt, e, 5, non, commissioned, officer, ...
2480    [government, relations, communications, and, o...
2481    [geek, squad, agent, professional, profile, it...
2482    [program, director, office, manager, summary, ...
2483    [storekeeper, ii, professional, summary, the, ...
Name: bag of words, Length: 2484, dtype: object

In [7]:
data_keywords = [
    # Core Roles & Job Titles
    "data scientist", "senior data scientist", "machine learning engineer", "applied scientist",
    "research scientist", "data analyst", "business analyst", "bi analyst", "analytics engineer",
    "data engineer", "etl developer", "big data engineer", "mlops engineer", "ml engineer",
    "ai engineer", "decision scientist", "quantitative analyst", "quant analyst", "statistician",
    "data science intern", "data engineer intern", "analytics intern",

    # Programming Languages
    "python", "r", "sql", "scala", "java", "julia", "c", "c++", "c#", "matlab", "javascript",
    "typescript", "sas", "stata",

    # Python Data & ML Ecosystem
    "numpy", "pandas", "scipy", "scikit-learn", "sklearn", "statsmodels", "tensorflow", "keras",
    "pytorch", "jax", "xgboost", "lightgbm", "catboost", "prophet", "fbprophet", "pymc", "pymc3",
    "pymc4", "pyspark", "sparklyr", "dask", "ray", "rapids", "cudf", "cuml",

    # R Ecosystem
    "tidyverse", "dplyr", "tidyr", "ggplot2", "data.table", "caret", "mlr", "shiny", "rmarkdown",

    # Data Visualization & BI Tools
    "matplotlib", "seaborn", "plotly", "bokeh", "altair", "ggplot", "tableau", "tableau server",
    "tableau prep", "power bi", "looker", "looker studio", "mode analytics", "qlik", "qlikview",
    "qlik sense", "superset", "metabase", "redash", "grafana", "kibana", "excel dashboards",
    "google data studio",

    # Databases & Query Engines
    "mysql", "postgresql", "postgres", "sql server", "oracle", "sqlite", "snowflake", "bigquery",
    "amazon redshift", "redshift", "azure synapse", "teradata", "vertica", "mariadb", "db2",
    "presto", "trino", "hive", "impala", "clickhouse", "elasticsearch", "mongodb", "cassandra",
    "redis", "dynamodb", "neo4j",

    # Big Data & Distributed Computing
    "hadoop", "hdfs", "mapreduce", "spark", "apache spark", "spark sql", "spark streaming",
    "kafka", "apache kafka", "flink", "apache flink", "storm", "kinesis", "beam", "apache beam",
    "airflow on spark", "emr", "databricks", "cloudera", "hortonworks",

    # Cloud Platforms & Managed Data Services
    "amazon web services", "aws", "google cloud platform", "gcp", "microsoft azure", "azure",
    "cloud computing", "serverless", "aws s3", "s3", "aws lambda", "aws glue", "aws athena",
    "aws emr", "aws sagemaker", "azure data lake", "azure databricks", "azure ml",
    "azure functions", "google bigquery", "google cloud storage", "gcs", "google cloud functions",
    "vertex ai", "google cloud composer", "gcp dataflow", "gcp dataproc",
    "snowflake on aws", "snowflake on azure", "snowflake on gcp",

    # Data Engineering, ETL, and Workflow Orchestration
    "etl", "elt", "data pipelines", "batch processing", "streaming pipelines", "real-time data",
    "cdc", "change data capture", "data warehouse", "data lake", "data lakehouse", "data modeling",
    "dimensional modeling", "star schema", "snowflake schema", "kimball", "inmon", "olap", "oltp",
    "data integration", "data ingestion", "data transformation", "data normalization",
    "denormalization", "data quality", "data governance", "data lineage",
    "master data management", "mdm", "apache airflow", "airflow", "luigi", "prefect", "dagster",
    "dbt", "dbt core", "dbt cloud", "fivetran", "stitch", "talend", "informatica", "ssis",
    "data factory",

    # Machine Learning Fundamentals & Techniques
    "supervised learning", "unsupervised learning", "reinforcement learning", "semi-supervised learning",
    "self-supervised learning", "classification", "regression", "clustering", "anomaly detection",
    "outlier detection", "recommendation systems", "recommender systems", "ranking",
    "time series forecasting", "forecasting", "dimensionality reduction", "feature selection",
    "feature extraction", "cross-validation", "hyperparameter tuning", "model selection",
    "ensemble methods", "bagging", "boosting", "stacking", "random forests", "gradient boosting",
    "k-nearest neighbors", "knn", "support vector machines", "svm", "logistic regression",
    "linear regression", "ridge regression", "lasso", "elastic net", "decision trees",
    "naive bayes", "k-means", "hierarchical clustering", "dbscan", "pca", "t-sne", "umap",
    "arima", "sarima", "var", "prophet", "survival analysis",

    # Deep Learning & Specialized ML
    "neural networks", "deep learning", "cnn", "convolutional neural networks", "rnn",
    "recurrent neural networks", "lstm", "gru", "transformers", "attention mechanisms", "gan",
    "generative adversarial networks", "autoencoders", "variational autoencoders", "vae",
    "sequence models", "sequence-to-sequence", "seq2seq", "encoder-decoder", "computer vision",
    "image classification", "object detection", "segmentation", "opencv", "yolo", "mask r-cnn",
    "nlp", "natural language processing", "text classification", "named entity recognition", "ner",
    "sentiment analysis", "topic modeling", "lda", "word embeddings", "word2vec", "glove",
    "bert", "gpt", "large language models", "llms", "speech recognition",
    "recommendation engines", "collaborative filtering", "content-based filtering", "bandits",
    "contextual bandits", "reinforcement learning",

    # Statistics, Mathematics & Experimentation
    "descriptive statistics", "inferential statistics", "probability", "probability distributions",
    "hypothesis testing", "a/b testing", "experimentation", "statistical modeling",
    "regression analysis", "anova", "nonparametric tests", "confidence intervals", "p-values",
    "bayesian statistics", "bayesian inference", "mcmc", "markov chain monte carlo",
    "monte carlo simulation", "maximum likelihood estimation", "mle", "sampling methods",
    "bootstrapping", "time series analysis", "stochastic processes", "linear algebra",
    "optimization", "convex optimization",

    # MLOps, Model Serving & Productionization
    "mlops", "model deployment", "model serving", "production machine learning", "model monitoring",
    "model lifecycle", "ml pipelines", "ml orchestration", "ml platforms", "ci/cd",
    "continuous integration", "continuous delivery", "docker", "containers", "kubernetes", "k8s",
    "kubeflow", "mlflow", "experiment tracking", "model registry", "feature store", "feast",
    "tecton", "pipeline automation", "batch scoring", "online scoring", "shadow deployment",
    "canary deployment", "model drift", "data drift", "model explainability",
    "explainable ai", "xai", "shap", "lime", "fairness", "responsible ai",

    # General Software Engineering & DevOps Skills
    "git", "github", "gitlab", "bitbucket", "version control", "branching", "pull requests",
    "code review", "unit testing", "integration testing", "pytest", "unittest",
    "test-driven development", "tdd", "software engineering", "clean code", "design patterns",
    "rest apis", "building apis", "flask", "django", "fastapi", "microservices",
    "command line", "linux", "unix", "bash", "shell scripting", "jenkins",
    "github actions", "circleci", "azure devops", "terraform", "infrastructure as code",
    "monitoring", "logging", "prometheus", "elk stack",

    # Data Cleaning, Wrangling & Feature Engineering
    "data cleaning", "data wrangling", "data preprocessing", "data munging",
    "missing data imputation", "outlier handling", "feature engineering", "feature scaling",
    "normalization", "standardization", "encoding categorical variables", "one-hot encoding",
    "label encoding", "target encoding", "feature importance", "text preprocessing",
    "tokenization", "stemming", "lemmatization", "stop word removal", "n-grams",
    "regular expressions", "regex",

    # Analytics, Business, and Domain Keywords
    "business intelligence", "descriptive analytics", "diagnostic analytics", "predictive analytics",
    "prescriptive analytics", "kpi", "kpis", "metrics", "dashboards", "reporting",
    "performance metrics", "customer analytics", "marketing analytics", "product analytics",
    "growth analytics", "financial analytics", "operations analytics", "risk analytics",
    "churn analysis", "customer segmentation", "cohort analysis", "funnel analysis",
    "pricing analytics", "supply chain analytics", "logistics optimization", "fraud detection",
    "credit risk", "marketing attribution", "experimentation platform", "stakeholder management",
    "data storytelling", "data-driven decision making",

    # Common Tools for Analysts
    "excel", "microsoft excel", "google sheets", "pivot tables", "vlookup", "index match",
    "excel macros", "vba", "jupyter", "jupyter notebook", "jupyterlab", "rstudio", "vs code",
    "spyder", "sql queries", "ad-hoc analysis", "exploratory data analysis", "eda",
    "reporting automation",

    # Soft Skills & Impact Phrases
    "communication skills", "strong communication", "presenting insights", "stakeholder communication",
    "cross-functional collaboration", "working with product managers", "working with engineers",
    "working with business stakeholders", "problem solving", "critical thinking",
    "analytical mindset", "attention to detail", "self-starter", "fast learner", "leadership",
    "mentoring", "leading projects", "project management", "agile", "scrum", "kanban",

    # Education & Certifications Keywords
    "bachelor's degree in computer science", "bachelor's in statistics",
    "bachelor's in mathematics", "bachelor's in engineering",
    "master's degree in data science", "master's in computer science",
    "master's in statistics", "phd", "coursera", "edx", "udacity", "datacamp", "kaggle",
    "aws certified machine learning", "aws certified data analytics",
    "google professional data engineer",
    "google professional machine learning engineer",
    "microsoft certified data scientist",
    "databricks certification"
]

data_keywords = list(set(data_keywords))

len(data_keywords)

523

In [8]:
# from sklearn.feature_extraction.text import CountVectorizer

# vectorizer = CountVectorizer(
#     lowercase=True,
#     stop_words="english" ,
#     # built‑in English stopwords
#     # you can also customize token_pattern or analyzer if you want
#     vocabulary=data_keywords
# )
# # vectorizer.fit(data_keywords)
# X = vectorizer.transform(df_job["Job Description"])  # X is bag‑of‑words matrix
# feature_names = vectorizer.get_feature_names_out()

In [40]:
df_resume.columns

Index(['ID', 'Resume_str', 'Resume_html', 'Category', 'bag of words'], dtype='object')

In [42]:


corpus = []


for doc in df_resume['Resume_str']:
    corpus.append(doc)
        
for doc in df_job['Job Description']:
    corpus.append(doc)
 
        
len(corpus)

4761

In [43]:
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(corpus)

In [44]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1124910 stored elements and shape (4761, 45871)>

In [45]:
job_vecs = X[:-1]      # all rows except last
query_vec = X[-1]      # last row = query resume

# 4. Compute similarity from query resume to each job
sims = cosine_similarity(query_vec, job_vecs)[0]
sims.sum()

227.9601553267628

In [46]:
import numpy as np
ranked_indices = np.argsort(-sims) 
ranked_indices

array([3808, 4751, 3329, ..., 4225,  656, 4151])

In [47]:
ranked_indices

array([3808, 4751, 3329, ..., 4225,  656, 4151])

In [48]:
X[3808]

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 112 stored elements and shape (1, 45871)>

In [ ]:
#PLEASE DISREGARD, I have no idea how to do tfidf